# 生成式 AI 應用開發：第 10 週 Embedding 與語意搜尋實作

本週承接第 9 週的文件處理與 chunking，進一步把文字片段轉成 embedding，並用相似度搜尋找出和使用者問題最相關的內容。

本週只做到 retrieval：找資料、排序、檢查來源。第 11 週才會把搜尋結果組成 context，交給模型產生 RAG 回答。

## 本週學習目標

1. 說明 embedding 為什麼能讓文字被搜尋與比較。
2. 用本機假 embedding 跑通向量索引流程，不依賴 API key。
3. 使用 cosine similarity 建立最小語意搜尋器。
4. 理解 OpenAI Embeddings API 的呼叫位置與安全界線。
5. 比較最小索引、ChromaDB 與 FAISS 在教學與專題中的取捨。
6. 將 notebook 練習銜接到 `week10_semantic_search_app/` Streamlit 專案。

In [ ]:
# 若在 Colab 或新環境執行，先取消下一行註解安裝套件。
# 這些套件支援本週的向量運算、OpenAI Embeddings API 與後續 Streamlit 專案。
# %pip install numpy openai python-dotenv pandas pypdf python-docx streamlit

import hashlib
import json
import math
import os
from pprint import pprint

import numpy as np

## 從第 9 週接到第 10 週

第 9 週的核心任務是把不同文件格式整理成乾淨文字，並切成 chunks。第 10 週不重新處理 PDF 或 Word reader，而是把「已經切好的 chunk」轉成可搜尋的資料結構。

你可以把資料流記成：

`原始文件 -> 文字抽取 -> 清理 -> chunking -> embedding -> 相似度搜尋 -> 檢索結果`

其中前半段是第 9 週，後半段是第 10 週。

In [ ]:
# 課堂使用的小型知識庫：每一筆都模擬第 9 週 chunking 後的輸出。
# 真實專案會從 PDF、DOCX、CSV、TXT 或 MD 抽取文字後再切成 chunks。
sample_chunks = [
    {
        "chunk_id": 0,
        "source": "ai_course_faq.md",
        "start": 0,
        "end": 90,
        "text": "第 9 週會教文件抽取、文字清理與 chunking，讓長文件變成可處理的小段落。",
    },
    {
        "chunk_id": 1,
        "source": "ai_course_faq.md",
        "start": 91,
        "end": 175,
        "text": "第 10 週會把 chunks 轉成 embedding，並用 cosine similarity 做語意搜尋。",
    },
    {
        "chunk_id": 2,
        "source": "ai_course_faq.md",
        "start": 176,
        "end": 265,
        "text": "第 11 週會把搜尋到的 chunks 組成 context，再交給模型回答，這就是 RAG 的基本流程。",
    },
    {
        "chunk_id": 3,
        "source": "ai_course_faq.md",
        "start": 266,
        "end": 360,
        "text": "Streamlit 專案應該把 API key 放在環境變數或 secrets，不要寫進程式碼或推到 GitHub。",
    },
]

pprint(sample_chunks)

## Embedding 是什麼？

Embedding 是把文字轉成一串數字向量。向量中的每個數字不是給人閱讀的標籤，而是模型用來表示文字特徵的位置。

對應到應用開發時，我們關心的是三件事：

- 相近意思的文字，在向量空間中通常比較接近。
- 查詢文字也可以轉成同一種 embedding，才能和文件 chunks 比較。
- embedding 本身不是答案，它只是 retrieval 階段的搜尋訊號。

In [ ]:
def local_demo_embed(text: str, dimensions: int = 64) -> list[float]:
    """用本機規則產生固定長度的假 embedding。

    參數：
        text: 要轉成向量的文字。
        dimensions: 向量維度；所有文字都必須使用同一個維度才能比較。

    回傳：
        長度固定的 float list。

    教學重點：
    - 這個函式不呼叫外部 API，適合課堂快速測試資料流。
    - 它只用字元與詞彙雜湊製造向量，不具備真正語意理解。
    - 真正專題效果需要改用 OpenAI Embeddings API 或其他 embedding model。
    """
    vector = np.zeros(dimensions, dtype=float)
    normalized = text.lower().strip()
    chars = [char for char in normalized if not char.isspace()]
    features = list(chars)
    features += [chars[index] + chars[index + 1] for index in range(len(chars) - 1)]
    features += normalized.split()

    if not features:
        return vector.tolist()

    for feature in features:
        digest = hashlib.md5(feature.encode("utf-8")).hexdigest()
        bucket = int(digest, 16) % dimensions
        vector[bucket] += 1.0

    norm = np.linalg.norm(vector)
    if norm == 0:
        return vector.tolist()
    return (vector / norm).tolist()


demo_vector = local_demo_embed("語意搜尋會比較查詢與文件片段的向量")
print("向量長度：", len(demo_vector))
print("前 8 個數值：", [round(value, 3) for value in demo_vector[:8]])

## 為什麼要計算 cosine similarity？

向量搜尋需要一個分數判斷「查詢」和「文件片段」有多接近。cosine similarity 會看兩個向量方向是否相近，常見範圍約在 -1 到 1 之間。

在本週案例中，我們會把分數越高解讀成越相關。但要注意：分數門檻不是永遠固定，會受到 embedding model、資料品質、chunk 大小與查詢寫法影響。

In [ ]:
def cosine_similarity(vec_a: list[float], vec_b: list[float]) -> float:
    """計算兩個向量的 cosine similarity。

    參數：
        vec_a: 第一個向量，通常是查詢文字的 embedding。
        vec_b: 第二個向量，通常是某個文件 chunk 的 embedding。

    回傳：
        兩個向量的方向相似度；任一向量長度為 0 時回傳 0。

    教學重點：
    - 先檢查零向量，避免除以 0。
    - 將 list 轉成 numpy array，讓點積與長度計算更清楚。
    """
    a = np.asarray(vec_a, dtype=float)
    b = np.asarray(vec_b, dtype=float)
    norm_a = np.linalg.norm(a)
    norm_b = np.linalg.norm(b)
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return float(np.dot(a, b) / (norm_a * norm_b))


query_vec = local_demo_embed("第 10 週要做語意搜尋")
chunk_vec = local_demo_embed(sample_chunks[1]["text"])
print("相似度：", round(cosine_similarity(query_vec, chunk_vec), 4))

In [ ]:
# 先用三句話觀察排序結果。這不是正式搜尋器，只是確認相似度概念。
query = "如何把文件片段變成可以搜尋的向量？"
query_vector = local_demo_embed(query)

scores = []
for chunk in sample_chunks:
    chunk_vector = local_demo_embed(chunk["text"])
    scores.append({
        "chunk_id": chunk["chunk_id"],
        "score": round(cosine_similarity(query_vector, chunk_vector), 4),
        "text": chunk["text"],
    })

scores.sort(key=lambda item: item["score"], reverse=True)
pprint(scores)

## 建立最小語意搜尋器

在正式導入 ChromaDB 或 FAISS 前，先用 `list[dict]` 建立最小索引最適合教學。原因是每一筆資料都看得到：

- `text`：原始 chunk 文字。
- `source`、`start`、`end`：來源與位置，方便後續引用。
- `embedding`：用來計算相似度的向量。

只要學生能讀懂這個資料結構，再看向量資料庫就不會只是在背套件用法。

In [ ]:
def embed_texts(texts: list[str], *, offline: bool = True, model: str | None = None) -> list[list[float]]:
    """批次把多段文字轉成 embedding。

    參數：
        texts: 要轉向量的文字清單。
        offline: True 時使用本機假 embedding；False 時才呼叫 OpenAI API。
        model: OpenAI embedding model 名稱，離線模式不會使用。

    回傳：
        與 `texts` 順序一致的向量清單。

    可能錯誤：
        ValueError: 輸入不是非空文字清單。

    教學重點：
    - 用同一個函式包住離線與 API 模式，讓 app 主流程不用知道細節。
    - 本 cell 先只實作離線模式；OpenAI API 版本會在後面補上。
    """
    if not texts or any(not text.strip() for text in texts):
        raise ValueError("texts 必須是非空文字清單。")

    if offline:
        return [local_demo_embed(text) for text in texts]

    raise NotImplementedError("OpenAI Embeddings API 版本會在後面章節實作。")

In [ ]:
def build_embedding_index(chunks: list[dict], *, offline: bool = True, model: str | None = None) -> list[dict]:
    """把 chunks 加上 embedding，形成最小可搜尋索引。

    參數：
        chunks: 第 9 週 chunking 產生的資料，每筆至少要有 `text`。
        offline: True 時使用本機假 embedding；False 時使用真實 Embeddings API。
        model: 真實 API 模式使用的 embedding model。

    回傳：
        每筆 chunk 都多出 `embedding` 欄位的新 list。

    教學重點：
    - 不直接修改原始 chunks，避免後續 debug 時分不清資料從哪一步改變。
    - `zip(chunks, vectors)` 假設 embedding 回傳順序與輸入文字順序一致。
    """
    if not chunks:
        raise ValueError("沒有 chunks，無法建立索引。")

    texts = [chunk["text"] for chunk in chunks]
    vectors = embed_texts(texts, offline=offline, model=model)

    indexed = []
    for chunk, vector in zip(chunks, vectors):
        item = dict(chunk)
        item["embedding"] = vector
        indexed.append(item)
    return indexed


index = build_embedding_index(sample_chunks)
print("索引筆數：", len(index))
print("第一筆索引欄位：", index[0].keys())

In [ ]:
def search_index(
    query: str,
    indexed_chunks: list[dict],
    *,
    top_k: int = 3,
    offline: bool = True,
    model: str | None = None,
) -> list[dict]:
    """使用 query embedding 搜尋最相近的 chunks。

    參數：
        query: 使用者輸入的查詢文字。
        indexed_chunks: 已經含有 embedding 的 chunks。
        top_k: 回傳前幾筆最高分結果。
        offline: query embedding 要使用離線模式或真實 API。
        model: 真實 API 模式使用的 embedding model。

    回傳：
        依相似度由高到低排序的搜尋結果，不包含原始 embedding。

    教學重點：
    - 搜尋時也要把 query 轉成同一種 embedding。
    - 回傳給 UI 的資料移除 embedding，避免畫面顯示大量數字。
    """
    if not query.strip():
        raise ValueError("查詢內容不可為空。")
    if top_k <= 0:
        raise ValueError("top_k 必須大於 0。")

    query_vector = embed_texts([query], offline=offline, model=model)[0]
    results = []
    for chunk in indexed_chunks:
        score = cosine_similarity(query_vector, chunk["embedding"])
        payload = {key: value for key, value in chunk.items() if key != "embedding"}
        payload["score"] = round(score, 4)
        results.append(payload)

    results.sort(key=lambda item: item["score"], reverse=True)
    return results[:top_k]


pprint(search_index("第 11 週會做什麼？", index, top_k=2))

In [ ]:
def run_local_checks() -> None:
    """用不花錢的方式檢查本週核心資料流。

    教學重點：
    - smoke test 不追求完整測試框架，而是確認課堂最容易壞的環節。
    - 若這個 cell 失敗，先修正本機資料流，再考慮接 OpenAI API。
    """
    local_index = build_embedding_index(sample_chunks)
    assert len(local_index) == len(sample_chunks)
    assert "embedding" in local_index[0]

    results = search_index("語意搜尋與 embedding", local_index, top_k=2)
    assert len(results) == 2
    assert results[0]["score"] >= results[1]["score"]
    assert "embedding" not in results[0]

    print("本機 embedding、索引與搜尋流程檢查通過。")


run_local_checks()

## 接 OpenAI Embeddings API 前的安全界線

真實 embedding 會產生更好的語意搜尋效果，但它是付費 API，而且可能會把文件內容送到外部服務。因此專題中要先確認：

- 不把真實 API key 寫進 notebook、程式碼或 GitHub。
- 上傳文件不得含有不該送出教室或公司環境的敏感資料。
- 本機流程先用假 embedding 確認 UI、chunking 與排序沒有問題。
- 只有在使用者明確按下按鈕或切換設定時，才呼叫付費 API。

官方 Embeddings 文件：https://platform.openai.com/docs/guides/embeddings

In [ ]:
DEFAULT_EMBEDDING_MODEL = "text-embedding-3-small"


def get_openai_embedding_model() -> str:
    """讀取 embedding model 名稱，保留環境變數覆蓋彈性。"""
    return os.getenv("OPENAI_EMBEDDING_MODEL", DEFAULT_EMBEDDING_MODEL)


def embed_texts_with_openai(texts: list[str], model: str | None = None) -> list[list[float]]:
    """呼叫 OpenAI Embeddings API，把文字批次轉成真實語意向量。

    參數：
        texts: 要轉向量的文字清單。
        model: 指定 embedding model；未指定時使用環境變數或預設值。

    回傳：
        與 `texts` 順序一致的 embedding list。

    可能錯誤：
        ValueError: 輸入清單為空或含空字串。
        RuntimeError: 缺少 API key 或 API 呼叫失敗。

    教學重點：
    - API key 只從環境變數讀取，不寫在 notebook 裡。
    - Embeddings API 可以批次處理多段文字，通常比逐段呼叫更好管理。
    - 這個函式只在付費示範 cell 主動開啟時才會執行。
    """
    if not texts or any(not text.strip() for text in texts):
        raise ValueError("texts 必須是非空文字清單。")

    api_key = os.getenv("OPENAI_API_KEY")
    if not api_key:
        raise RuntimeError("找不到 OPENAI_API_KEY，請先設定環境變數。")

    from openai import OpenAI

    client = OpenAI(api_key=api_key)
    selected_model = model or get_openai_embedding_model()
    try:
        response = client.embeddings.create(model=selected_model, input=texts)
    except Exception as exc:
        raise RuntimeError(f"呼叫 Embeddings API 失敗：{exc}") from exc

    return [item.embedding for item in response.data]

In [ ]:
RUN_PAID_EMBEDDING_DEMO = False

if RUN_PAID_EMBEDDING_DEMO:
    # 只有在你已設定測試用 OPENAI_API_KEY，並確認文件內容可送出時，才改成 True。
    real_vectors = embed_texts_with_openai([chunk["text"] for chunk in sample_chunks])
    print("真實 embedding 筆數：", len(real_vectors))
    print("第一筆向量維度：", len(real_vectors[0]))
else:
    print("付費 Embeddings API 示範目前關閉；本週可先用離線模式完成流程。")

## ChromaDB、FAISS 與最小索引怎麼選？

| 選項 | 適合情境 | 教學優點 | 需要注意 |
|---|---|---|---|
| `list[dict]` + NumPy | 第一個語意搜尋原型、課堂解釋資料流 | 每筆資料都看得到，debug 最直覺 | 資料量變大會慢，沒有持久化 |
| ChromaDB | 小型知識庫、需要保存文字與 metadata | API 接近文件資料庫，方便銜接 RAG 引用來源 | 多一個服務/套件概念，部署時要處理儲存位置 |
| FAISS | 大量向量、重視查詢速度 | 向量索引能力強，效能好 | 需自行管理原文與 metadata，Windows 安裝較容易卡住 |

本週正式教材把 ChromaDB 與 FAISS 放在「理解與選型」層級；實作主線先用最小索引，確保學生知道向量資料庫到底替他們做了哪些事。

In [ ]:
vector_db_notes = [
    {
        "tool": "最小索引",
        "when_to_use": "課堂第一版與小型 demo",
        "main_tradeoff": "容易理解，但不適合大量資料或持久化查詢",
    },
    {
        "tool": "ChromaDB",
        "when_to_use": "第 11 週 RAG 小型知識庫與來源引用",
        "main_tradeoff": "metadata 管理方便，但增加套件與部署複雜度",
    },
    {
        "tool": "FAISS",
        "when_to_use": "資料量較大且查詢效能是主要問題",
        "main_tradeoff": "速度快，但原文與 metadata 要另外保存",
    },
]

pprint(vector_db_notes)

## 選讀：ChromaDB preview

前面的最小索引讓我們看懂資料流；ChromaDB 則示範「向量資料庫」會幫我們管理哪些事情。

這個 preview 使用已經建立好的離線 `index`，不需要 OpenAI API key。若環境尚未安裝 ChromaDB，先取消下一行註解安裝：

```python
# %pip install chromadb
```

請注意：這裡仍是選讀，不是本週主線。正式主線先用最小索引理解 retrieval；第 11 週 RAG 才會更需要 ChromaDB 這類工具管理來源引用與 metadata。

In [ ]:
def preview_chromadb_query(indexed_chunks: list[dict], query: str, top_k: int = 2) -> list[dict]:
    """用 ChromaDB in-memory collection 示範向量資料庫查詢流程。

    參數：
        indexed_chunks: 已經由 `build_embedding_index()` 建好的 chunks，每筆都要有 embedding。
        query: 使用者查詢文字；此函式會用同一個離線 embedding 規則轉成 query vector。
        top_k: 要回傳的最相近片段數量。

    回傳：
        ChromaDB 查詢結果整理後的 list；若未安裝 ChromaDB，回傳空 list。

    教學重點：
    - `collection.add()` 會同時放入 ids、documents、embeddings 與 metadatas。
    - `collection.query()` 回傳的 distance 不是直接的 similarity；cosine 空間下可用 `1 - distance` 做近似觀察。
    - 這裡用 `EphemeralClient()`，資料只存在記憶體，適合課堂 preview；正式專題若要保存資料，才需要改成持久化設定。
    """
    try:
        import chromadb
    except ImportError:
        print("尚未安裝 chromadb；若要執行 preview，請先取消上一個 markdown cell 的安裝指令。")
        return []

    if not indexed_chunks:
        raise ValueError("indexed_chunks 不可為空。")
    if top_k <= 0:
        raise ValueError("top_k 必須大於 0。")

    # EphemeralClient 不會把資料寫到磁碟，適合課堂示範與測試。
    client = chromadb.EphemeralClient()
    collection_name = "week10_preview"

    # 同一個 notebook cell 重複執行時，先刪除舊 collection，避免 id 重複。
    try:
        client.delete_collection(collection_name)
    except Exception:
        pass

    collection = client.create_collection(
        name=collection_name,
        metadata={"hnsw:space": "cosine"},
    )
    collection.add(
        ids=[str(chunk["chunk_id"]) for chunk in indexed_chunks],
        documents=[chunk["text"] for chunk in indexed_chunks],
        embeddings=[chunk["embedding"] for chunk in indexed_chunks],
        metadatas=[
            {
                "chunk_id": int(chunk.get("chunk_id", index)),
                "source": str(chunk.get("source", "unknown")),
                "start": int(chunk.get("start", 0)),
                "end": int(chunk.get("end", 0)),
            }
            for index, chunk in enumerate(indexed_chunks)
        ],
    )

    query_vector = local_demo_embed(query)
    raw = collection.query(
        query_embeddings=[query_vector],
        n_results=min(top_k, len(indexed_chunks)),
        include=["documents", "metadatas", "distances"],
    )

    results = []
    documents = raw["documents"][0]
    metadatas = raw["metadatas"][0]
    distances = raw["distances"][0]
    for document, metadata, distance in zip(documents, metadatas, distances):
        # ChromaDB 在 cosine 空間回傳 distance；這裡轉成相似度讓學生能和前面的分數比較。
        results.append({
            "chunk_id": metadata.get("chunk_id"),
            "source": metadata.get("source"),
            "score": round(1 - float(distance), 4),
            "text": document,
        })
    return results


chroma_preview_results = preview_chromadb_query(index, "第 11 週會做什麼？", top_k=2)
if chroma_preview_results:
    pprint(chroma_preview_results)

## 對應到 Week10 Streamlit 專案

正式專案位置：

`week10/week10_semantic_search_app/`

建議執行流程：

```bash
cd week10/week10_semantic_search_app
python -m venv .venv
.\.venv\Scripts\Activate.ps1
python -m pip install --upgrade pip
pip install -r requirements.txt
streamlit run app.py
```

教學時先保持離線示範模式，確認文件上傳、chunking、建立索引與搜尋結果都能運作；最後再示範如何切換成 OpenAI Embeddings。

## 練習 A：準備可放進索引的 chunks

`build_embedding_index()` 需要每筆 chunk 至少有 `text`，但 Streamlit 結果頁還需要來源與位置。教師版示範保留既有 metadata，缺少時才補預設值。

In [ ]:
def prepare_chunks_for_embedding(chunks: list[dict], source_name: str) -> list[dict]:
    """補齊 chunk metadata，讓搜尋結果能顯示來源。

    參數：
        chunks: 第 9 週 chunking 的輸出。
        source_name: 文件來源名稱，例如檔名。

    回傳：
        每筆都含有 `chunk_id`、`source`、`start`、`end`、`text` 的 list。

    教學重點：
    - metadata 不是裝飾；第 11 週引用來源時會直接依賴它。
    - 若原本 chunks 已有 start/end，應優先保留原值。
    """
    prepared = []
    for index, chunk in enumerate(chunks):
        text = str(chunk.get("text", "")).strip()
        if not text:
            continue
        item = {
            "chunk_id": int(chunk.get("chunk_id", index)),
            "source": str(chunk.get("source", source_name)),
            "start": int(chunk.get("start", 0)),
            "end": int(chunk.get("end", len(text))),
            "text": text,
        }
        prepared.append(item)
    return prepared


prepared_chunks = prepare_chunks_for_embedding(sample_chunks, "ai_course_faq.md")
pprint(prepared_chunks[:2])

## 練習 B：用分數門檻過濾搜尋結果

搜尋 top-k 不代表每筆結果都值得交給後續 RAG。教師版示範只保留 `score >= min_score` 的結果。

In [ ]:
def filter_results_by_threshold(results: list[dict], min_score: float) -> list[dict]:
    """依相似度門檻過濾搜尋結果。

    參數：
        results: `search_index()` 回傳的搜尋結果。
        min_score: 最低可接受相似度。

    回傳：
        分數達標的結果清單。

    教學重點：
    - 門檻太高可能沒有資料可用，門檻太低可能把不相關內容送進 RAG。
    - 實務上門檻要用測試問題與人工檢查一起調整。
    """
    return [item for item in results if item.get("score", 0) >= min_score]


raw_results = search_index("API key 要放在哪裡？", index, top_k=4)
filtered_results = filter_results_by_threshold(raw_results, min_score=0.15)
pprint(filtered_results)

## 練習 C：組成 RAG 前的 context

第 10 週不直接生成答案，但可以先練習把搜尋結果整理成第 11 週會使用的 context 字串。教師版示範控制長度並保留來源資訊。

In [ ]:
def build_rag_context(results: list[dict], max_chars: int = 900) -> str:
    """把搜尋結果整理成 RAG 可使用的 context 字串。

    參數：
        results: 語意搜尋結果，應包含 source、chunk_id、score 與 text。
        max_chars: context 最長字元數，避免把過多內容送進模型。

    回傳：
        含來源標記的 context 字串。

    教學重點：
    - context 不是越長越好；過長會增加成本，也可能稀釋重點。
    - 來源標記要跟 chunk 一起保存，方便第 11 週做引用。
    """
    blocks = []
    used_chars = 0
    for result in results:
        block = (
            f"[來源：{result.get('source', 'unknown')} | "
            f"chunk {result.get('chunk_id', '?')} | "
            f"score {result.get('score', 0):.4f}]\n"
            f"{result.get('text', '').strip()}"
        )
        next_length = used_chars + len(block) + (2 if blocks else 0)
        if next_length > max_chars:
            break
        blocks.append(block)
        used_chars = next_length
    return "\n\n".join(blocks)


context_preview = build_rag_context(raw_results, max_chars=500)
print(context_preview)

## App 挑戰：把 notebook 概念搬到 Streamlit

請開啟 `week10/week10_semantic_search_app/`，觀察以下對應關係：

- `document_utils.py`：第 9 週文件處理與 chunking。
- `embedding_utils.py`：本週 embedding、索引與搜尋。
- `app.py`：按鈕觸發索引建立、搜尋表單與結果顯示。

教師可示範同一份文件在不同 chunk size、overlap 與 top-k 下的排序差異。

In [ ]:
project_checklist = {
    "can_upload_sample_file": True,
    "can_build_offline_index": True,
    "can_search_top_k": True,
    "knows_when_to_use_openai_embeddings": True,
}

pprint(project_checklist)

## 完成檢核

- 能畫出第 9 週到第 10 週的資料流。
- 能說明 embedding、query embedding、chunk embedding 與 cosine similarity 的關係。
- 能使用離線模式建立索引並搜尋 top-k 結果。
- 能說明為什麼 API key 不應寫在 notebook 或 GitHub。
- 能比較最小索引、ChromaDB 與 FAISS 的適用情境。
- 能開啟 Week10 Streamlit 專案並操作一次完整搜尋流程。

## 常見問題

Q：離線假 embedding 可以交專題嗎？

A：可以用來展示流程，但不適合宣稱具備真正語意搜尋能力。正式專題若要搜尋品質，應使用真實 embedding model。

Q：本週一定要使用 ChromaDB 或 FAISS 嗎？

A：不一定。第 10 週重點是理解 embedding 與 retrieval。ChromaDB 與 FAISS 是當資料量、持久化或效能需求出現後的工具選項。

Q：為什麼第 10 週不直接讓 AI 回答？

A：先把 retrieval 做穩，才能在第 11 週清楚檢查 RAG 回答到底根據哪些來源。

## 下一週預告：RAG 問答

第 11 週會把本週搜尋出的 chunks 組成 context，要求模型只能根據 context 回答，並在回答中保留來源。也就是把「找到相關資料」進一步變成「根據資料回答問題」。